# Data Analysis & Modeling: Predict Next-Week Play Time

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
Interview format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic.

## The interviewer prompt

You work on a game platform. At the start of day 0, the team wants to estimate how many minutes each player will spend playing during the next seven days. You receive a player table and a session log.

Use only information available before day 0. Predict **next_week_minutes**, the total minutes during days 0–6. For example, a prediction of 75 means an estimated 75 minutes over that week. A player who does not play has a valid target of zero.

**Problem 1 — Data analysis (14 minutes).** Load both CSVs, inspect data quality, clean the session log, and build one row per player. Report the fraction of players without historical sessions and average observed historical play time by device.

**Problem 2 — Modeling (21 minutes).** Select features, create a train/test split, fit a simple regression model and a constant baseline, evaluate their errors, and predict for a new player. Explain your choices as you code. No neural network or from-scratch optimizer is required.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the data |
| 9–14 min | Analyze and prepare features |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? What exactly is the target?

**Practice version.** Replace each NotImplementedError cell with your code, then run its supplied check cell. Imports, inputs and checks are provided. Run All intentionally stops at unfinished exercises. Use the matching solution notebook only when reviewing.

## Data contract — supplied by the interviewer

**players.csv:** one row per player. Player attributes are snapshots taken before day 0. The export was assembled after every player's outcome window finished, so the target is known for training. Keep every player, including those without historical sessions.

| Column | Meaning |
| --- | --- |
| user_id | Unique player identifier; used for joins, not as a numeric predictor |
| account_age_days | Account age at the prediction cutoff; may be missing |
| country | Country category; may be missing |
| device | Device category; may be missing |
| next_week_minutes | Target: total minutes played during days 0–6; zero is valid |
| returned_next_week | Whether the player returns during days 0–6; a future outcome, not an input feature |

**sessions.csv:** one row per logged session before cleaning. A player can have multiple sessions or none.

| Column | Meaning |
| --- | --- |
| session_id | Session identifier; the export may repeat identical rows |
| user_id | Player identifier; always refers to a player in players.csv |
| day_offset | Day relative to the cutoff: -1 means yesterday, 0 means the prediction day; only -7 through -1 are valid history |
| minutes_played | Session duration; missing or a negative sentinel means unknown, not zero |
| completed | 1 if the session was completed, otherwise 0; remains valid when duration is unknown |

**Aggregation rules for this exercise:**

- Keep sessions whose durations are unknown: they still count as sessions.
- With no historical sessions, set session_count, known_duration_count and total_minutes to 0; leave avg_minutes and completion_rate missing.
- With sessions but no known durations, leave total_minutes and avg_minutes missing.
- With some known durations, total_minutes sums the observed durations only; it can understate true play time.

Distinct players at this shared cutoff are assumed to be independent samples for the mock. The random holdout estimates performance on other players from this population; it does not measure robustness to changes over future weeks.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter and select **.venv-codesignal/bin/python**. No network, GPU or additional package installation is required.

The two input files are **data/codesignal_roblox/players.csv** and **data/codesignal_roblox/sessions.csv**. The provided cell imports libraries and sets DATA_DIR, a pathlib.Path directory object. In Step 1, use pd.read_csv(DATA_DIR / "players.csv") and pd.read_csv(DATA_DIR / "sessions.csv") to load them. Do not modify the source CSVs.

Start the 35-minute coding timer after setup.


In [64]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression, SGDRegressor
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, f1_score, log_loss, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, precision_score, recall_score)
SEED = 42

DATA_DIR = Path("data/codesignal_roblox")
assert (DATA_DIR / "players.csv").exists(), "Run with the workspace root as your working directory."


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Load **players** and **sessions** with pd.read_csv. Inspect their shapes, sample rows, column types and missing values. Check unique player IDs, exact duplicate sessions, negative durations and records outside days -7 through -1.

Inspect **next_week_minutes** with describe, including the 50th, 90th and 99th percentiles. Report the fraction of labels equal to zero. A high upper percentile can reveal a long tail: a small number of players with much larger values.

**Say before coding:** "I will first check what each row represents, which fields are available at prediction time, and whether the target contains missing or invalid values."

**Possible interviewer questions**

1. What does one training example represent, and what is its target?
2. Can returned_next_week be a feature?
3. What should you inspect before dropping missing rows?


In [25]:
# Create the variables named in the task above.
players = pd.read_csv(DATA_DIR / "players.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")

display(players)
#print(players.shape)
display(sessions)
#print(sessions.shape)

#print(players.isna().sum())

#print(sessions.duplicated().sum())

#print(~sessions["day_offset"].between(-7,-1))

#print(sessions["minutes_played"].lt(0))
display(players["next_week_minutes"].describe(percentiles=[0.5, 0.9, 0.99]))

print("zero target fraction", players["next_week_minutes"].eq(0).mean())


#raise NotImplementedError("Complete step 1, then run the checks below.")


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes
0,user_0001,184.0,US,mobile,0,0.00
1,user_0002,6.0,US,mobile,0,0.00
2,user_0003,56.0,US,desktop,0,0.00
3,user_0004,39.0,US,desktop,1,247.43
4,user_0005,295.0,GB,NaN,1,5.96
...,...,...,...,...,...,...
995,user_0996,192.0,IN,mobile,1,187.33
996,user_0997,543.0,US,desktop,0,0.00
997,user_0998,6.0,US,mobile,0,0.00
998,user_0999,77.0,BR,mobile,0,0.00


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1
...,...,...,...,...,...
4960,session_03545,user_0719,-7,25.42,1
4961,session_00709,user_0147,-3,-1.00,1
4962,session_02745,user_0547,-2,54.67,1
4963,session_01861,user_0378,-7,46.00,1


count    1000.000000
mean       43.316520
std        82.095041
min         0.000000
50%         0.000000
90%       192.697000
99%       290.693100
max       299.290000
Name: next_week_minutes, dtype: float64

zero target fraction 0.707


In [ ]:
assert players["user_id"].is_unique
assert players["next_week_minutes"].notna().all()
assert players["next_week_minutes"].ge(0).all()
assert sessions["user_id"].isin(players["user_id"]).all()


## Problem 1 · Step 2 — Clean the data (4–9 min)

Create **clean_sessions** without modifying **sessions**. Remove exact duplicate rows, retain day_offset values from -7 through -1 inclusive, and replace negative minutes_played with NaN. Keep rows with unknown durations.

After removing exact duplicates, check that session_id is unique. Conflicting records with the same ID require investigation rather than silently keeping an arbitrary row. Do not fill missing values with learned statistics yet.

**Say before coding:** "I am applying fixed validity rules, not learning statistics from the full dataset."

**Possible interviewer questions**

1. Why keep a session whose duration is missing?
2. Why not replace missing durations with zero?
3. Why postpone median imputation until after splitting?


In [42]:
clean_sessions = sessions.drop_duplicates().copy()

#
print(clean_sessions.duplicated().sum())

clean_sessions = clean_sessions.loc[clean_sessions["day_offset"].between(-7,-1)].copy()
print(sessions.shape)
print(clean_sessions["day_offset"].between(-7,-1).sum())
## check
display(clean_sessions)
print(clean_sessions["session_id"].is_unique)
mask = clean_sessions["minutes_played"].lt(0)

clean_sessions.loc[mask,"minutes_played"] = np.nan

display(clean_sessions.loc[mask])

0
(4965, 5)
4905


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1
...,...,...,...,...,...
4960,session_03545,user_0719,-7,25.42,1
4961,session_00709,user_0147,-3,-1.00,1
4962,session_02745,user_0547,-2,54.67,1
4963,session_01861,user_0378,-7,46.00,1


True


,session_id,user_id,day_offset,minutes_played,completed
77,session_02669,user_0525,-6,NaN,0
394,session_00766,user_0158,-5,NaN,1
481,session_03472,user_0700,-6,NaN,1
515,session_03425,user_0691,-4,NaN,1
523,session_03738,user_0759,-3,NaN,0
669,session_02620,user_0518,-2,NaN,0
688,session_04066,user_0833,-5,NaN,1
831,session_01957,user_0397,-7,NaN,0
1115,session_02385,user_0471,-5,NaN,0
1184,session_02960,user_0592,-6,NaN,0


In [ ]:
assert clean_sessions["day_offset"].between(-7, -1).all()
assert clean_sessions["minutes_played"].dropna().ge(0).all()
assert clean_sessions["session_id"].is_unique


## Problem 1 · Step 3 — Analyze and prepare features (9–14 min)

Group **clean_sessions** by user_id and create **activity** with:

- **session_count:** number of sessions, including unknown durations.
- **known_duration_count:** number of nonmissing durations.
- **total_minutes:** sum of observed durations; missing if every duration is unknown.
- **avg_minutes:** mean of observed durations.
- **completion_rate:** mean of the 0/1 completed indicator.

Left join activity into players to create **features**, one row per player. Create **no_history**, a Boolean mask for players without matching sessions, and apply the aggregation rules in the data contract.

Report the no-history fraction. Group features by device and report player count, count of observed totals, and mean observed total_minutes. Include players without history and retain a missing-device group.

**Say before coding:** "I need one training row per player, including people without any history."

**Possible interviewer questions**

1. Why a left join?
2. Can a partial total understate activity?
3. How do you check that a merge did not multiply rows?


In [51]:
# Create the variables named in the task above.
actvity = clean_sessions.groupby("user_id").agg(
    session_count = ("session_id", "size"),
    known_duration_count = ("minutes_played", "count"),
    total_minutes = ("minutes_played", "sum"),
    avg_minutes = ("minutes_played", "mean"),
    completion_rate = ("completed", "mean"),
)

actvity.loc[actvity["known_duration_count"].eq(0), "total_minutes"] = np.nan

#display(actvity)
features = players.merge(actvity, on="user_id", how="left", validate="one_to_one")
display(features)

no_history = features["session_count"].isna()
#display(no_history)
features.loc[no_history, ["session_count", "known_duration_count", "total_minutes"]] = 0

display(features)





#raise NotImplementedError("Complete step 3, then run the checks below.")


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes,session_count,known_duration_count,total_minutes,avg_minutes,completion_rate
0,user_0001,184.0,US,mobile,0,0.00,NaN,NaN,NaN,NaN,NaN
1,user_0002,6.0,US,mobile,0,0.00,3.0,0.0,NaN,NaN,0.666667
2,user_0003,56.0,US,desktop,0,0.00,1.0,1.0,59.11,59.110000,1.000000
3,user_0004,39.0,US,desktop,1,247.43,4.0,4.0,138.60,34.650000,0.500000
4,user_0005,295.0,GB,NaN,1,5.96,9.0,9.0,342.79,38.087778,0.333333
...,...,...,...,...,...,...,...,...,...,...,...
995,user_0996,192.0,IN,mobile,1,187.33,6.0,6.0,242.85,40.475000,0.833333
996,user_0997,543.0,US,desktop,0,0.00,8.0,8.0,315.98,39.497500,0.500000
997,user_0998,6.0,US,mobile,0,0.00,2.0,2.0,27.84,13.920000,1.000000
998,user_0999,77.0,BR,mobile,0,0.00,10.0,9.0,416.01,46.223333,0.500000


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes,session_count,known_duration_count,total_minutes,avg_minutes,completion_rate
0,user_0001,184.0,US,mobile,0,0.00,0.0,0.0,0.00,NaN,NaN
1,user_0002,6.0,US,mobile,0,0.00,3.0,0.0,NaN,NaN,0.666667
2,user_0003,56.0,US,desktop,0,0.00,1.0,1.0,59.11,59.110000,1.000000
3,user_0004,39.0,US,desktop,1,247.43,4.0,4.0,138.60,34.650000,0.500000
4,user_0005,295.0,GB,NaN,1,5.96,9.0,9.0,342.79,38.087778,0.333333
...,...,...,...,...,...,...,...,...,...,...,...
995,user_0996,192.0,IN,mobile,1,187.33,6.0,6.0,242.85,40.475000,0.833333
996,user_0997,543.0,US,desktop,0,0.00,8.0,8.0,315.98,39.497500,0.500000
997,user_0998,6.0,US,mobile,0,0.00,2.0,2.0,27.84,13.920000,1.000000
998,user_0999,77.0,BR,mobile,0,0.00,10.0,9.0,416.01,46.223333,0.500000


In [ ]:
assert len(features) == len(players)
assert features["user_id"].is_unique
assert features.loc[no_history, "total_minutes"].eq(0).all()
unknown = features["session_count"].gt(0) & features["known_duration_count"].eq(0)
assert features.loc[unknown, "total_minutes"].isna().all()


## Problem 2 · Step 4 — Select features and split (14–18 min)

Define **numeric_features** as account_age_days, session_count, known_duration_count, total_minutes, avg_minutes and completion_rate. Define **categorical_features** as country and device.

Create **X** using only these columns and **y** from next_week_minutes. Exclude user_id and both future-outcome columns from X. Create **X_train**, **X_test**, **y_train**, **y_test** with an 80/20 train_test_split and random_state=SEED. Do not stratify directly on the continuous target.

**Say before coding:** "I will keep the holdout untouched and learn preprocessing only on training players."

**Possible interviewer questions**

1. Should we stratify by this continuous target?
2. What if rows contain repeated cutoffs for the same user?
3. How do you check that X and y still correspond?


In [54]:
# Create the variables named in the task above.
numeric_features = ["account_age_days","session_count",
                    "known_duration_count","total_minutes",
                     "avg_minutes", "completion_rate" ]

categorical_features = [ "country", "device"]

X = features[numeric_features + categorical_features]
y = features["next_week_minutes"]

X_train, X_test, y_train, y_test = train_test_split(X,y ,test_size=0.2, random_state=SEED)

display(X_train)
display(X_test)
display(y_train)
display(y_test)



#raise NotImplementedError("Complete step 4, then run the checks below.")


,account_age_days,session_count,known_duration_count,total_minutes,avg_minutes,completion_rate,country,device
29,24.0,3.0,3.0,102.51,34.170000,0.666667,IN,desktop
535,272.0,9.0,9.0,501.47,55.718889,0.777778,CA,mobile
695,182.0,4.0,4.0,58.25,14.562500,0.000000,US,tablet
557,9.0,10.0,10.0,110.18,11.018000,0.300000,CA,NaN
836,41.0,9.0,9.0,128.37,14.263333,0.555556,US,mobile
...,...,...,...,...,...,...,...,...
106,403.0,10.0,10.0,300.39,30.039000,0.700000,BR,mobile
270,107.0,3.0,3.0,146.75,48.916667,0.666667,BR,desktop
860,126.0,3.0,3.0,75.50,25.166667,0.666667,US,desktop
435,180.0,9.0,9.0,128.82,14.313333,0.444444,US,desktop


,account_age_days,session_count,known_duration_count,total_minutes,avg_minutes,completion_rate,country,device
521,2.0,10.0,10.0,458.43,45.843000,0.800000,US,desktop
737,211.0,3.0,3.0,96.22,32.073333,0.666667,CA,mobile
740,43.0,9.0,9.0,497.69,55.298889,0.777778,GB,mobile
660,33.0,2.0,2.0,46.40,23.200000,0.500000,BR,desktop
411,137.0,8.0,7.0,269.10,38.442857,0.125000,US,desktop
...,...,...,...,...,...,...,...,...
408,105.0,0.0,0.0,0.00,NaN,NaN,IN,tablet
332,629.0,3.0,3.0,48.96,16.320000,0.000000,CA,desktop
208,374.0,7.0,6.0,213.29,35.548333,0.285714,US,desktop
613,39.0,5.0,5.0,85.74,17.148000,0.200000,CA,tablet


29       0.00
535     91.62
695      0.00
557      0.00
836      0.00
        ...  
106      0.00
270      0.00
860      0.00
435    192.41
102      0.00
Name: next_week_minutes, Length: 800, dtype: float64

521    299.29
737    241.78
740      9.80
660      0.00
411    284.97
        ...  
408      0.00
332      0.00
208      0.00
613      0.00
78     290.69
Name: next_week_minutes, Length: 200, dtype: float64

In [ ]:
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert not {"user_id", "returned_next_week", "next_week_minutes"} & set(X.columns)


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Fit **baseline**, a DummyRegressor(strategy="mean"), which predicts the training-label mean for every player.

Create **numeric_pipeline** with SimpleImputer(strategy="median") followed by StandardScaler. Create **categorical_pipeline** with SimpleImputer(strategy="most_frequent") followed by OneHotEncoder(handle_unknown="ignore"). Combine them in a ColumnTransformer named **preprocess**.

Create **model**, a Pipeline with preprocess and a regressor named **regressor**, using Ridge(alpha=1.0). Fit only X_train and y_train. Ridge predicts a weighted sum of the transformed features plus an intercept; its L2 penalty discourages large coefficients.

**Say before coding:** "I will compare a constant training-mean baseline with a regularized linear model. The pipeline will learn imputation, scaling and encoding from training data only."

**Possible interviewer questions**

1. Why scale for Ridge?
2. Is the mean the best constant for every metric?
3. What happens when you call model.fit on the Pipeline?


In [ ]:
# Create the variables named in the task above.

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("normalization", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])


preprocess = ColumnTransformer([
    ("num_pipe", numeric_pipeline, numeric_features),
    ("cat_pipe", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocess", preprocess ),
    ("regressor", Ridge(alpha=1.0)),
])

model.fit(X=X_train,y=y_train)

#print(model.steps)

#raise NotImplementedError("Complete step 5, then run the checks below.")


-- Epoch 1
Norm: 30.13, NNZs: 15, Bias: 26.975939, T: 800, Avg. loss: 3041.831643
Total training time: 0.00 seconds.
-- Epoch 2
Norm: 28.32, NNZs: 15, Bias: 25.326465, T: 1600, Avg. loss: 2885.244713
Total training time: 0.00 seconds.
-- Epoch 3
Norm: 27.40, NNZs: 15, Bias: 25.232521, T: 2400, Avg. loss: 2873.653384
Total training time: 0.00 seconds.
-- Epoch 4
Norm: 29.38, NNZs: 15, Bias: 25.644385, T: 3200, Avg. loss: 2885.626214
Total training time: 0.00 seconds.
-- Epoch 5
Norm: 32.58, NNZs: 15, Bias: 27.695765, T: 4000, Avg. loss: 2872.599652
Total training time: 0.00 seconds.
-- Epoch 6
Norm: 29.88, NNZs: 15, Bias: 26.129768, T: 4800, Avg. loss: 2868.620341
Total training time: 0.00 seconds.
-- Epoch 7
Norm: 31.19, NNZs: 15, Bias: 26.539583, T: 5600, Avg. loss: 2876.673191
Total training time: 0.00 seconds.
-- Epoch 8
Norm: 32.63, NNZs: 15, Bias: 26.571059, T: 6400, Avg. loss: 2873.972083
Total training time: 0.00 seconds.
-- Epoch 9
Norm: 32.00, NNZs: 15, Bias: 26.355319, T: 720

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num_pipe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('normalization',
                                                                   StandardScaler())]),
                                                  ['account_age_days',
                                                   'session_count',
                                                   'known_duration_count',
                                                   'total_minutes',
                                                   'avg_minutes',
                                                   'completion_rate']),
                                                 ('cat_pipe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['country', 'device'])])),
                ('regressor',
                 SGDRegressor(max_iter=20, random_state=42, tol=None,
                              verbose=1))])

In [ ]:
assert model.named_steps["regressor"].coef_.size > 0
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create **pred** and **baseline_pred** by predicting on X_test. The application requires nonnegative minutes, so apply np.maximum(prediction, 0) to both outputs as a fixed serving rule, not a choice tuned on test results.

Create a **scores** table with rows **mean_baseline** and **ridge** and columns **MAE_minutes**, **RMSE_minutes** and **R2**:

- **MAE:** average absolute prediction error, in minutes; lower is better.
- **RMSE:** square root of average squared error, in minutes; gives more weight to large errors; lower is better.
- **R2:** 1 minus model squared error divided by squared error around the test-label mean; higher is better. It can be negative.

Create **errors** containing actual_minutes, predicted_minutes and absolute_error. Display the five largest errors and give a short interpretation. Do not change hyperparameters based on this test set.

**Say before coding:** "I will report errors in minutes, compare against the constant baseline, and inspect where the model misses badly. I will keep model selection separate from final test evaluation."

**Possible interviewer questions**

1. Can R2 be negative?
2. What if the target has many zeros and a long tail?
3. Can you keep changing the model after looking at test metrics?


In [69]:
# Create the variables named in the task above.
pred = model.predict(X_test)
pred = np.maximum(pred,0)
def evaluate(gt, prediction):
    return { 
        "MAE_min": mean_absolute_error(gt,prediction),
        "RMSE_min": root_mean_squared_error(gt,prediction),
        "R2": r2_score(gt,prediction)
    }

scores = pd.DataFrame({
    "ridge": evaluate(y_test,pred)
    }).T

display(scores)



#raise NotImplementedError("Complete step 6, then run the checks below.")


,MAE_min,RMSE_min,R2
ridge,57.41832,77.857909,0.09346


In [ ]:
assert pred.shape == (len(y_test),)
assert np.isfinite(pred).all() and (pred >= 0).all()
np.testing.assert_allclose(scores.loc["ridge", "MAE_minutes"], np.abs(y_test.to_numpy() - pred).mean())
np.testing.assert_allclose(scores.loc["ridge", "RMSE_minutes"], np.sqrt(np.mean((y_test.to_numpy() - pred) ** 2)))


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

Create **new_player**, a one-row DataFrame with these already-aggregated pre-cutoff features:

| Feature | Value |
| --- | --- |
| account_age_days | np.nan |
| session_count | 3 |
| known_duration_count | 2 |
| total_minutes | 55 |
| avg_minutes | 27.5 |
| completion_rate | 2/3 |
| country | "UNSEEN_COUNTRY" |
| device | "mobile" |

Use the fitted model to create **new_prediction**, clip it at zero using the serving rule, and report the result in minutes. Do not fit again. Give a 30-second summary of the input features, split, model, evaluation, one limitation and one next experiment.

**Say before coding:** "Inference reuses the training transformations. An unseen category must not crash the pipeline."

**Possible interviewer questions**

1. Why not call fit_transform on the new player?
2. Does better predicted play time imply a better product?
3. What must training and inference share?


In [ ]:
# Create the variables named in the task above.
new_player = pd.DataFrame([{
    "account_age_days": np.nan, "session_count": 3, "known_duration_count": 2,
    "total_minutes": 55, "avg_minutes": 27.5, "completion_rate": 2/3,
    "country": "UNSEEN_COUNTRY", "device": "mobile",
}])

new_prediction = model.predict(new_player[numeric_features+categorical_features])
print("new_prediction",new_prediction)
#raise NotImplementedError("Complete step 7, then run the checks below.")


new_prediction [14.07318114]


In [ ]:
assert new_prediction.shape == (1,)
assert np.isfinite(new_prediction).all()


## If the interviewer extends the problem

These questions are optional, outside the 35-minute core.

1. **MAE is the business metric. Which constant baseline would you choose?** A training-label median minimizes training absolute error. Compare it with the training-mean baseline, which minimizes squared error.
2. **A few players have extremely large targets. What would you try?** Check that the values are valid, then consider a log1p target or a different model on validation data. Evaluate final predictions in minutes; exponentiating a predicted mean log value does not generally recover the mean in the original units.
3. **Does the model work equally well across devices?** Report per-device errors together with group counts. Small groups have uncertain estimates; avoid concluding that a device causes an error difference.
4. **How would you improve the model?** Try one tree-based model on a validation split, compare it with Ridge, and keep the final test set untouched until the choice is fixed.
5. **How would you evaluate future weeks?** Collect multiple prediction cutoffs, ensure every feature precedes its cutoff, and hold out a later period with fully observed outcome windows.


## References and provenance

This is an original synthetic regression exercise, not a reported Roblox question or an official CodeSignal assessment. The interview format follows the user-supplied **Data Analysis & Modeling Coding Interview** guide: 5 minutes introduction, 35 minutes for data analysis and model training, and 5 minutes Q&A. Step timings are suggested practice checkpoints.

The task, synthetic data and reference implementation are practice material. They do not describe actual Roblox player behavior or guarantee that this topic will be asked.

Implementation references: [scikit-learn model evaluation](https://scikit-learn.org/1.6/modules/model_evaluation.html), [avoiding data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage).
